# What's new in 0.2.5

This page runs one small example for each of the main additions in 0.2.5. Every
number on it is computed when the book is built, from simulated data with a
known answer, so it shows what the feature does rather than restating a
benchmark. The full list of changes is in the
[changelog](https://github.com/fcela/mboostrs/blob/main/CHANGELOG.md).

The examples are deliberately small so the page builds quickly. Speed-ups
measured on larger problems are quoted in the changelog, with the data set each
was measured on.

In [ ]:
import time
import warnings

import numpy as np
import polars as pl

import mboostrs as mb

rng = np.random.default_rng(0)

## Binned splines for non-Gaussian families

`bbs(x, bins=256)` replaces each covariate by the centre of one of 256
quantile bins before building the spline basis. The basis then has 256
distinct rows instead of one per observation, so each boosting step costs a
pass over the bins rather than over the data. Gaussian fits already had a fast
path that did not depend on the number of rows; binning is what brings the
other families — Binomial, Poisson, the distributional families — closer to it.
In 0.2.5 the specification search turns it on automatically for those families
once the data are large enough.

Below, a Binomial additive model on 40,000 rows is fitted with and without
binning, and both are scored on 10,000 held-out rows.

In [ ]:
n = 50_000
X = rng.standard_normal((n, 4))
eta = np.sin(2 * X[:, 0]) + X[:, 1] ** 2 / 2 - 0.5 + 0.5 * X[:, 2]
y = rng.binomial(1, 1 / (1 + np.exp(-eta))).astype(float)
binary = pl.DataFrame({f"x{j}": X[:, j] for j in range(4)}).with_columns(pl.Series("y", y))
train_b, test_b = binary[:40_000], binary[40_000:]


def log_loss(model, test):
    p = np.clip(np.asarray(model.predict(test, type="response", extrapolation="clip")).ravel(), 1e-6, 1 - 1e-6)
    obs = test["y"].to_numpy()
    return float(-np.mean(obs * np.log(p) + (1 - obs) * np.log(1 - p)))


fits = {}
for label, extra in [("exact", ""), ("binned", ", bins=256")]:
    formula = "y ~ " + " + ".join(f"bbs(x{j}, knot_placement='quantile'{extra})" for j in range(4))
    start = time.perf_counter()
    fits[label] = mb.gamboost(formula, data=train_b, family=mb.Binomial(),
                              control=mb.boost_control(300, 0.1))
    elapsed = time.perf_counter() - start
    print(f"{label:>6}: {elapsed:5.2f} s, held-out log-loss {log_loss(fits[label], test_b):.4f}")

The held-out log-loss is the same to three decimals: 256 bins are fine enough
that the curves the two fits find are indistinguishable at this sample size.
The time saved grows with the number of rows and of candidate terms, which is
why it matters most inside `mb.fit`, where the specification search fits many
candidate models. `binning_report()` records, for every learner, how many bins
were asked for and how many distinct values they replaced.

In [ ]:
display(fits["binned"].binning_report())

## Regularised trees

`btree` has two new controls, both off by default:

- `lambda_leaf` shrinks each leaf value towards zero by adding a ridge penalty
  to the leaf's denominator, as in XGBoost's `lambda`.
- `colsample` lets each tree look at only a random share of the candidate
  columns. The draw is seeded (`colsample_seed`), so a fit is reproducible.

Neither always helps. On the problem below — an interaction between two of six
columns — the unregularised tree is already well calibrated, and restricting
each tree to half the columns makes it miss the interaction more often. They
are tuning options, and the specification search treats them that way.

In [ ]:
p = 6
Xt = rng.standard_normal((2_000, p))
yt = (Xt[:, 0] * Xt[:, 1] + 0.5 * Xt[:, 2] + rng.normal(0, 0.5, 2_000) > 0).astype(float)
trees = pl.DataFrame({f"x{j}": Xt[:, j] for j in range(p)}).with_columns(pl.Series("y", yt))
train_t, test_t = trees[:1_500], trees[1_500:]
cols = ", ".join(f"x{j}" for j in range(p))

for label, extra in [
    ("no regularisation", ""),
    ("lambda_leaf=2", ", lambda_leaf=2.0"),
    ("lambda_leaf=2, colsample=0.5", ", lambda_leaf=2.0, colsample=0.5, colsample_seed=1"),
]:
    model = mb.gamboost(f"y ~ btree({cols}, maxdepth=4, split_method='histogram'{extra})",
                        data=train_t, family=mb.Binomial(), control=mb.boost_control(300, 0.1))
    print(f"{label:>30}: held-out log-loss {log_loss(model, test_t):.3f}")

## Several quantiles in one call: `quantile_boost`

`mb.quantile_boost(formula, data, quantiles=...)` cross-validates and fits one
quantile model per requested level in a single pass through the Rust core,
running the levels in parallel. Two things come with it that separate
`QuantReg` fits do not give you:

- the predicted quantiles are rearranged so they never cross; and
- `predict_interval(..., calibration=...)` widens or narrows the central
  interval on a held-out calibration set — conformalised quantile regression
  (CQR) — so its coverage holds in finite samples rather than only if every
  quantile model is right.

The data below have noise whose spread grows with `|x|`, so a good interval has
to be wider at the edges than in the middle.

In [ ]:
nq = 3_000
xq = rng.uniform(-2, 2, nq)
yq = np.sin(xq) + (0.3 + 0.3 * np.abs(xq)) * rng.standard_normal(nq)
quant = pl.DataFrame({"x": xq, "y": yq})
train_q, cal_q, test_q = quant[:2_000], quant[2_000:2_500], quant[2_500:]

mq = mb.quantile_boost("y ~ bbs(x)", train_q, quantiles=(0.05, 0.25, 0.5, 0.75, 0.95), seed=0)
interval = mq.predict_interval(test_q, level=0.90, calibration=cal_q)

obs = test_q["y"].to_numpy()
lower, upper = np.asarray(interval["lower"]), np.asarray(interval["upper"])
inside = (obs >= lower) & (obs <= upper)
edge = np.abs(test_q["x"].to_numpy()) > 1.5
print(f"90% CQR interval: coverage {inside.mean():.3f} overall")
print(f"mean width where |x| < 1.5: {np.mean((upper - lower)[~edge]):.2f}, "
      f"where |x| > 1.5: {np.mean((upper - lower)[edge]):.2f}")

`digest()` gives a short plain-text summary of a fitted object — here the
stopping iteration chosen for each quantile and its out-of-fold pinball loss.
Every fitted model type in 0.2.5 has one, and `mb.digest(obj)` calls it.

In [ ]:
print(mq.digest())

## Intervals that adapt when the noise changes

A split-conformal interval is calibrated once, on data from before deployment.
If the noise level then changes, its coverage changes with it and nothing in
the interval reacts. `mb.adaptive_conformal_stream` implements adaptive
conformal inference {cite}`gibbs2021adaptive`: after each new observation it
checks whether the last interval covered it, and moves the next interval's
target miscoverage up or down accordingly, so coverage over a run of
observations is pulled back towards the nominal level.

Here the noise standard deviation doubles at observation 800, right after the
training data end.

In [ ]:
nt = 1_200
tt = np.arange(nt)
xs = rng.uniform(-2, 2, nt)
noise_sd = np.where(tt < 800, 0.4, 0.8)
ys = np.sin(2 * xs) + noise_sd * rng.standard_normal(nt)
stream_data = pl.DataFrame({"t": tt, "x": xs, "y": ys})
before, after = stream_data[:800], stream_data[800:]

model_s = mb.gamboost("y ~ bbs(x, df=4)", data=before, control=mb.boost_control(100, 0.1))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    split = mb.conformal_interval(model_s, after, level=0.8, method="split", seed=0)

obs_s = after["y"].to_numpy()
predicted = np.asarray(model_s.predict(after)).ravel()
half_width = (np.asarray(split.upper) - np.asarray(split.lower)) / 2
stream = mb.adaptive_conformal_stream(predicted, obs_s, half_width, level=0.8, lr=0.05)


def coverage(lo, hi, rows=slice(None)):
    lo, hi = np.asarray(lo)[rows], np.asarray(hi)[rows]
    return float(np.mean((obs_s[rows] >= lo) & (obs_s[rows] <= hi)))


last = slice(-200, None)
print(f"nominal 80%; split conformal after the change: {coverage(split.lower, split.upper):.3f}")
print(f"adaptive stream, all 400 new rows:   {coverage(stream.lower, stream.upper):.3f}")
print(f"adaptive stream, last 200 new rows:  {coverage(stream.lower, stream.upper, last):.3f}")

The stream does not jump to 80% at once: it has to see misses before it widens,
and how fast it reacts is set by `lr`. A larger `lr` recovers sooner and
fluctuates more. `mb.fit(..., time="t")` records the time column and makes
`Fit.predict_interval()` default to the adaptive method, which weights recent
calibration residuals more heavily; that helps with gradual drift inside the
calibration window, not with a break after it, which is what the stream is
for.

## Causal effects: intervals for the effect at each point, and overlap weights

`mb.effect_model` estimates how a treatment effect varies with covariates (the
CATE). In 0.2.5, `interval_method="dr_orthogonal"` gives pointwise intervals
for that curve built from a doubly-robust pseudo-outcome, so the nuisance
models — propensity and outcome — enter only through a product of their
errors. In the package's Monte Carlo tests the 90% intervals cover the true effect
89.5–93.5% of the time, where the previous intervals covered 61–66%.

The example has a true effect `tau = 1 + x1`, so coverage can be checked
directly.

In [ ]:
nc = 2_000
x0 = rng.uniform(-2, 2, nc)
x1 = rng.normal(size=nc)
x2 = rng.normal(size=nc)
treated = rng.binomial(1, 1 / (1 + np.exp(-x0))).astype(float)
tau = 1.0 + x1
outcome = np.sin(x0) + 0.5 * x1 + 0.3 * x2 + tau * treated + rng.normal(0, 0.7, nc)
causal_data = {"t": treated, "y": outcome, "x0": x0, "x1": x1, "x2": x2}

spec = mb.CausalSpec(
    treatment="t", outcome="y", covariates=("x0", "x1", "x2"),
    time_zero="assignment", follow_up="end of study",
    adjustment_rationale="x0, x1 and x2 are measured before treatment",
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fx = mb.effect_model(causal_data, spec, terms="smooth", folds=5, seed=0,
                         interval_method="dr_orthogonal")
    band = fx.predict(causal_data, interval=True, level=0.90, output="numpy")

print(f"90% pointwise intervals cover the true effect at {np.mean((tau >= band['lower']) & (tau <= band['upper'])):.3f} of units")

When treated and untreated units barely overlap, the average treatment effect
over everyone (ATE) depends on extrapolating to people who almost never get
one of the two arms. `CausalSpec(estimand="ato")` targets the overlap
population instead, weighting each unit by `e(x) (1 - e(x))`, which needs no
propensity clipping. With `estimand="ate"`, `mb.aipw` now also computes the
overlap estimate when more than 5% of units sit outside the overlap region and
attaches it as `ato_companion`, so a thin-overlap ATE always comes with the
estimate that does not depend on that extrapolation.

In [ ]:
nc = 1_500
x0 = rng.uniform(-2, 2, nc)
x1 = rng.normal(size=nc)
propensity = 1 / (1 + np.exp(-(3.2 * x0 + 0.5 * x1)))  # many units near 0 or 1
treated = rng.binomial(1, propensity).astype(float)
outcome = 2 * x0 + x1 + (1 + 1.5 * x0) * treated + rng.normal(0, 0.8, nc)
thin = {"t": treated, "y": outcome, "x0": x0, "x1": x1}

spec_thin = mb.CausalSpec(
    treatment="t", outcome="y", covariates=("x0", "x1"), estimand="ate",
    time_zero="assignment", follow_up="end of study",
    adjustment_rationale="x0 and x1 are measured before treatment",
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ate = mb.aipw(thin, spec_thin, folds=4, smooth=True, tune=False, mstop=80, seed=3)

companion = ate.ato_companion
print(f"ATE: {ate.estimate:.2f} (se {ate.standard_error:.2f})")
if companion is not None:
    print(f"overlap-weighted companion: {companion['estimate']:.2f} (se {companion['se']:.2f}); "
          f"{companion['outside_overlap_share']:.0%} of units outside the overlap region")

The two estimates answer different questions, so they need not agree: the true
effect `1 + 1.5 x0` is larger where `x0` is large, and the overlap population
is concentrated near `x0 = 0`.

## Also in 0.2.5

- `mb.fit(..., uncertainty="conformal")` and `Fit.predict_interval()`; the
  decision trail now records a SHA-256 of the data, its schema and the software
  environment, and `mb.reproduce` replays `time=` and `uncertainty=`.
- The specification search no longer reads a bounded integer measurement —
  an age, a score — as category labels or a count response.
- Distributional fits try Student-t, log-normal and beta families alongside the
  Gaussian, chosen by out-of-fold CRPS.
- `python -m mboostrs guide <topic>` and the `families`, `learners`, `causal`, `conformal`, `diagnostics` and `quantile`
  submodules under `mboostrs`.
- The R package gains the new `bbs` and `btree` arguments, adaptive conformal
  intervals and `adaptive_conformal_stream`.